# Fig. S4: cell-type deconvolution

Works on a bulk RNA-seq gene count matrix of mouse brain tissue (for example `GSE338230_counts_gene_all.csv`)
and a sample sheet: reference-based deconvolution against the Allen Brain Cell Atlas whole-mouse-brain
taxonomy (cluster means of the MapMyCells reference), cell-type fractions of each sample, group comparisons,
and the OXPHOS-subunit content of each cell type in the reference.

**Input:** a gene count matrix (Ensembl gene IDs in the first column, one column per sample) and a CSV with
one row per sample: the sample name and its condition. The Allen reference files (about 1.4 GB) and the
Ensembl GTF are downloaded into `REF_DIR` on first use.

**Output** (in `OUT_DIR`): compartment and cell-type fractions of each sample (NNLS and nu-SVR), group
statistics, the OXPHOS content of each cell type (CSV), and the plots (PDF).

In [ ]:
"""Install the packages used here."""
%pip install -q h5py scikit-learn statsmodels

In [ ]:
"""Settings."""
from pathlib import Path

COUNTS = Path("GSE338230_counts_gene_all.csv")   # genes x samples
SAMPLES = Path("samples.csv")                     # one row per sample
OUT_DIR = Path("results/deconvolution")
REF_DIR = Path("reference")                       # the Allen reference and the GTF are downloaded here

SAMPLE_KEY = "sample"         # sample-sheet column holding the count-matrix column names
CONDITION_KEY = "condition"   # sample-sheet column with the groups
GROUP_ORDER = []              # plot order of the groups; [] = sorted
COMPARISONS = []              # (a, b) pairs of groups reported from Tukey's test, e.g. [("A", "B")]
COLOURS = {}                  # group -> colour in the dot plots; empty = default colours

In [ ]:
"""Analysis parameters (values used in the paper)."""
ABC = "https://allen-brain-cell-atlas.s3.us-west-2.amazonaws.com"
CFG = dict(
    stats=f"{ABC}/mapmycells/WMB-10X/20240831/precomputed_stats_ABC_revision_230821.h5",   # cluster means
    membership=f"{ABC}/metadata/WMB-taxonomy/20231215/cluster_to_cluster_annotation_membership.csv",
    pivoted=f"{ABC}/metadata/WMB-taxonomy/20231215/views/cluster_to_cluster_annotation_membership_pivoted.csv",
    gtf="https://ftp.ensembl.org/pub/release-109/gtf/mus_musculus/Mus_musculus.GRCm39.109.gtf.gz",  # lengths, symbols
    n_markers=40,          # marker genes per compartment
    min_log2fc=1.0,        # marker: at least 2-fold above every other compartment
    min_cpm=10,            # marker: expression in its own compartment (reference CPM)
    min_tpm=1,             # marker: mean bulk TPM
    nu=(0.25, 0.5, 0.75),  # nu-SVR cross-check (CIBERSORT)
)

In [ ]:
"""Imports."""
import gzip, json, re, urllib.request
import numpy as np
import pandas as pd
import h5py
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.optimize import nnls
from scipy.stats import f_oneway
from sklearn.svm import NuSVR
from statsmodels.stats.multicomp import pairwise_tukeyhsd

OUT_DIR.mkdir(parents=True, exist_ok=True)
REF_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
"""Library versions used in this run."""
import sys, platform
from importlib.metadata import version, PackageNotFoundError

print(f"{'python':<14}{sys.version.split()[0]} ({platform.platform()})")
for pkg in ["numpy", "pandas", "scipy", "h5py", "scikit-learn", "statsmodels", "matplotlib"]:
    try:
        print(f"{pkg:<14}{version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg:<14}not installed")

## Functions

In [ ]:
"""Plot style."""
mpl.rcParams.update({
    "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
    "font.size": 7, "axes.labelsize": 7, "axes.titlesize": 7,
    "xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 6,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 2, "ytick.major.size": 2,
    "axes.spines.top": False, "axes.spines.right": False,
})
MM = 1 / 25.4
CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]

In [ ]:
"""Downloads, gene table, reference compartments and cell types."""
def fetch(url):
    """Path of a reference file, downloaded once into REF_DIR."""
    path = REF_DIR / url.rsplit("/", 1)[-1]
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return path


def gene_table(gtf):
    """Gene name and exon-union length (bp) per Ensembl gene ID, from a GTF (path or URL)."""
    path = fetch(gtf) if str(gtf).startswith(("http://", "https://", "ftp://")) else Path(gtf)
    sym, exons = {}, {}
    with (gzip.open(path, "rt") if path.suffix == ".gz" else open(path)) as fh:
        for line in fh:
            f = line.split("\t", 8)
            if len(f) < 9 or f[2] not in ("gene", "exon"):
                continue
            gid = re.search(r'gene_id "([^"]+)"', f[8]).group(1)
            if f[2] == "gene":
                name = re.search(r'gene_name "([^"]+)"', f[8])
                sym[gid] = name.group(1) if name else gid
            else:
                exons.setdefault(gid, []).append((int(f[3]), int(f[4])))
    length = {}
    for gid, iv in exons.items():          # union of the exons of all transcripts of the gene
        iv.sort()
        total, (s0, e0) = 0, iv[0]
        for st, en in iv[1:]:
            if st > e0:
                total, s0, e0 = total + e0 - s0 + 1, st, en
            else:
                e0 = max(e0, en)
        length[gid] = total + e0 - s0 + 1
    return pd.DataFrame({"symbol": pd.Series(sym), "length": pd.Series(length)})


DG = {"037 DG Glut", "038 DG-PIR Ex IMN"}
CA = {"016 CA1-ProS Glut", "017 CA3 Glut", "025 CA2-FC-IG Glut", "023 SUB-ProS Glut", "031 CT SUB Glut", "033 NP SUB Glut"}


def compartment(cls, sub):
    """Reference compartment of a cluster, from its Allen class and subclass."""
    k = int(cls[:2])
    if sub in DG: return "Neurons: DG granule/IMN"
    if sub in CA: return "Neurons: CA/SUB"
    if k in (1, 2): return "Neurons: IT-ET, NP-CT-L6b Glut"
    if k in (6, 7): return "Neurons: CTX-CGE/MGE GABA"
    if k == 9: return "Neurons: CNU-LGE GABA"
    if k == 8: return "Neurons: CNU-MGE GABA"
    if k == 10: return "Neurons: LSX GABA"
    if k in (3, 5): return "Neurons: OB"
    if k in range(11, 17): return "Neurons: HY, CNU-HYa"
    if k == 17: return "Neurons: MH-LH Glut"
    if k == 18: return "Neurons: TH Glut"
    if k in range(19, 30): return "Neurons: MB, HB, CB"
    if sub == "319 Astro-TE NN": return "Astrocytes: TE"
    if sub == "325 CHOR NN": return "Choroid plexus"
    if sub in ("322 Tanycyte NN", "323 Ependymal NN", "324 Hypendymal NN"): return "Ependymal"
    if k in (30, 32): return "Astrocytes: other"
    if sub == "326 OPC NN": return "OPC"
    if sub == "327 Oligo NN": return "Oligodendrocytes"
    if sub in ("331 Peri NN", "332 SMC NN", "333 Endo NN"): return "Vascular"
    if sub in ("329 ABC NN", "330 VLMC NN"): return "Fibroblasts"
    if sub == "334 Microglia NN": return "Microglia"
    return "BAM & other immune"


CLASSES = {   # cell types shown in Fig. S4, as sums of compartments
    "Neurons": lambda c: c.startswith("Neurons"),
    "Astrocytes": lambda c: c.startswith("Astrocytes"),
    "Oligodendrocytes": lambda c: c == "Oligodendrocytes",
    "OPC": lambda c: c == "OPC",
    "Microglia": lambda c: c == "Microglia",
    "BAM & other immune": lambda c: c == "BAM & other immune",
    "Vascular": lambda c: c == "Vascular",
    "Fibroblasts, choroid plexus & ependymal": lambda c: c in ("Fibroblasts", "Choroid plexus", "Ependymal"),
}
OXPHOS = r"^(Nduf[abcsv]\d+|Sdh[abcd]|Uqcr(c1|c2|fs1|h|b|q|10|11)|Cyc1|Cox(4i1|5a|5b|6a1|6b1|6c|7a2|7b|7c|8a)|Atp5.*)$"

## Reference

The MapMyCells reference holds the mean log2(CPM + 1) expression of each of the 5,322 clusters of the
whole-mouse-brain taxonomy. Cluster means are back-transformed to CPM and averaged, weighted by the number
of cells of each cluster, into 22 compartments: neuronal populations grouped by Allen class, and
non-neuronal cell types.

In [ ]:
"""Compartment profiles (CPM) from the Allen cluster means."""
mem = pd.read_csv(fetch(CFG["membership"]))
mem = mem[mem.cluster_annotation_term_set_name == "cluster"].set_index("cluster_annotation_term_label")
piv = pd.read_csv(fetch(CFG["pivoted"])).set_index("cluster_alias")
with h5py.File(fetch(CFG["stats"]), "r") as f:
    genes = json.loads(f["col_names"][()])
    cl = pd.DataFrame({"row": pd.Series(json.loads(f["cluster_to_row"][()]))})
    n = f["n_cells"][:]
    cl["alias"] = mem.loc[cl.index, "cluster_alias"].values
    cl["cells"] = mem.loc[cl.index, "number_of_cells"].values
    cl = cl.join(piv[["class", "subclass"]], on="alias")
    cl["compartment"] = [compartment(c, s) for c, s in zip(cl["class"], cl["subclass"])]
    REF = {}
    for comp, d in cl.groupby("compartment"):
        d = d.sort_values("row")
        lin = np.expm1(f["sum"][d.row.values, :] / n[d.row.values, None] * np.log(2))
        REF[comp] = (lin * d.cells.values[:, None]).sum(0) / d.cells.sum()
REF = pd.DataFrame(REF, index=genes)
print(cl.groupby("compartment").cells.agg(["count", "sum"]).rename(columns={"count": "clusters", "sum": "cells"}).to_string())

## Deconvolution

Bulk counts are converted to TPM with exon-union gene lengths, because the 10x reference counts molecules.
For each compartment, up to 40 marker genes are selected (at least 2-fold above every other compartment,
at least 10 CPM in the compartment, mean bulk TPM at least 1). The fraction of mRNA contributed by each
compartment is estimated by non-negative least squares on the marker matrix, with every gene scaled by its
mean across compartments; a CIBERSORT-style nu-support vector regression on the same markers is the
cross-check.

In [ ]:
"""Bulk TPM, marker genes, NNLS fractions and the nu-SVR cross-check."""
meta = pd.read_csv(SAMPLES, dtype={SAMPLE_KEY: str, CONDITION_KEY: str}).set_index(SAMPLE_KEY)
counts = pd.read_csv(COUNTS, index_col=0)[meta.index]
gt = gene_table(CFG["gtf"])
rpk = counts.div(gt.length.reindex(counts.index), axis=0).dropna()
tpm = rpk / rpk.sum() * 1e6
ref = REF.loc[REF.index.intersection(tpm.index)]

detected = tpm.loc[ref.index].mean(axis=1) >= CFG["min_tpm"]
markers = {}
for c in ref.columns:
    lfc = np.log2((ref[c] + 1) / (ref.drop(columns=c).max(axis=1) + 1))
    ok = (lfc >= CFG["min_log2fc"]) & (ref[c] >= CFG["min_cpm"]) & detected
    markers[c] = lfc[ok].sort_values(ascending=False).index[: CFG["n_markers"]].tolist()
mk = sorted(set(sum(markers.values(), [])))
M = ref.loc[mk]
scale = 1 / M.mean(axis=1)
A = M.mul(scale, axis=0).values
FRAC = pd.DataFrame({s: (lambda p: p / p.sum())(nnls(A, tpm.loc[mk, s].mul(scale).values)[0]) for s in tpm.columns},
                    index=ref.columns).T

Az = (M.values - M.values.mean()) / M.values.std()
svr = {}
for s in tpm.columns:
    b = tpm.loc[mk, s].values
    bz = (b - b.mean()) / b.std()
    best = None
    for nu in CFG["nu"]:
        w = np.clip(NuSVR(kernel="linear", nu=nu, C=1.0).fit(Az, bz).coef_.ravel(), 0, None)
        rmse = np.sqrt(np.mean((Az @ w - bz) ** 2)) if w.sum() > 0 else np.inf
        if best is None or rmse < best[0]:
            best = (rmse, w)
    svr[s] = best[1] / best[1].sum()
SVR = pd.DataFrame(svr, index=ref.columns).T
FRAC.to_csv(OUT_DIR / "fractions_compartments_nnls.csv")
SVR.to_csv(OUT_DIR / "fractions_compartments_nusvr.csv")
print(f"{len(mk)} marker genes, {len(FRAC)} samples")

In [ ]:
"""Cell-type fractions (% of mRNA), one-way ANOVA and Tukey's test."""
groups = GROUP_ORDER or sorted(meta[CONDITION_KEY].unique())
order = [s for g in groups for s in meta.index[meta[CONDITION_KEY] == g]]
cond = meta.loc[order, CONDITION_KEY]
total = lambda F: pd.DataFrame({k: F.loc[order, [c for c in F.columns if f(c)]].sum(axis=1) for k, f in CLASSES.items()}) * 100
CT, CT_SVR = total(FRAC), total(SVR)
CT.assign(**{CONDITION_KEY: cond}).to_csv(OUT_DIR / "cell_type_fractions_pct.csv")

rows = []
for k in CLASSES:
    y = CT[k]
    tk = pairwise_tukeyhsd(y.values, cond.values)
    tab = pd.DataFrame(tk.summary().data[1:], columns=tk.summary().data[0])
    row = {"cell_type": k, "ANOVA_P": f_oneway(*[y[cond == g] for g in groups]).pvalue,
           "r_NNLS_vs_nuSVR": np.corrcoef(CT[k], CT_SVR[k])[0, 1]}
    row.update({f"mean_{g}": y[cond == g].mean() for g in groups})
    for a, b in COMPARISONS:
        hit = tab[((tab.group1 == a) & (tab.group2 == b)) | ((tab.group1 == b) & (tab.group2 == a))]
        row[f"Tukey_P_{a}_vs_{b}"] = float(hit["p-adj"].iloc[0])
    rows.append(row)
STATS = pd.DataFrame(rows)
STATS.to_csv(OUT_DIR / "cell_type_stats.csv", index=False)
print(STATS.round(3).to_string(index=False))

In [ ]:
"""OXPHOS content of each cell type: share of its reference mRNA from nuclear-encoded subunits of complexes I-V."""
sym = gt.symbol.reindex(ref.index).fillna("")
ox = ref.loc[sym.str.match(OXPHOS)].sum() / ref.sum() * 100
w = FRAC.mean()                 # cell types made of several compartments: weighted by their mean fractions
OX = pd.Series({k: float((ox[cs] * w[cs]).sum() / w[cs].sum()) if w[cs].sum() > 0 else float(ox[cs].mean())
                for k, f in CLASSES.items() for cs in [[c for c in ref.columns if f(c)]]})
OX.rename("OXPHOS_pct_of_mRNA").to_csv(OUT_DIR / "cell_type_oxphos_content.csv")
print(OX.round(2).to_string())

## Plots

In [ ]:
"""Fig. S4: composition of each sample, cell types by group, OXPHOS content of each cell type."""
col = {g: COLOURS.get(g, f"C{i}") for i, g in enumerate(groups)}
fig = plt.figure(figsize=(183 * MM, 178 * MM))

ax = fig.add_axes([0.08, 0.70, 0.56, 0.26])
bottom = np.zeros(len(order))
for k, c in zip(CLASSES, CAT):
    ax.bar(range(len(order)), CT[k].values, bottom=bottom, color=c, width=0.82, edgecolor="white", linewidth=0.4, label=k)
    bottom += CT[k].values
ax.set(xlim=(-0.6, len(order) - 0.4), ylim=(0, 100), ylabel="% of mRNA", title="Cell-type composition per sample")
ax.set_xticks(range(len(order)), order, rotation=90, fontsize=5)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.34), ncol=4, frameon=False, fontsize=5.5)
i = 0
for g in groups:
    n = int((cond == g).sum())
    ax.text(i + (n - 1) / 2, -0.2, g, transform=ax.get_xaxis_transform(), ha="center", va="top", fontsize=6)
    i += n

rng = np.random.default_rng(3)
st = STATS.set_index("cell_type")
for j, k in enumerate(CLASSES):
    r_, c_ = divmod(j, 4)
    a2 = fig.add_axes([0.08 + c_ * 0.235, 0.34 - r_ * 0.235, 0.17, 0.155])
    for x, g in enumerate(groups):
        v = CT.loc[cond == g, k].values
        a2.scatter(x + rng.uniform(-0.12, 0.12, len(v)), v, s=10, color=col[g], zorder=3)
        a2.plot([x - 0.25, x + 0.25], [v.mean()] * 2, color="0.1", lw=1.0, zorder=4)
    top, span, lvl = CT[k].max(), max(CT[k].max() - CT[k].min(), 1e-6), 0
    for a, b in COMPARISONS:
        p = st.loc[k, f"Tukey_P_{a}_vs_{b}"]
        if p < 0.05:
            xa, xb = groups.index(a), groups.index(b)
            yb = top + span * (0.10 + 0.17 * lvl)
            a2.plot([xa, xa, xb, xb], [yb - span * 0.03, yb, yb, yb - span * 0.03], color="0.3", lw=0.6)
            a2.text((xa + xb) / 2, yb, f"P = {p:.3f}" if p >= 0.001 else "P < 0.001", ha="center", va="bottom", fontsize=5)
            lvl += 1
    a2.set_ylim(CT[k].min() - span * 0.1, top + span * (0.25 + 0.17 * lvl))
    a2.set_xticks(range(len(groups)), groups if r_ == 1 else [], rotation=45, ha="right", fontsize=5.5)
    a2.set_title(k, fontsize=6.5)
    if c_ == 0:
        a2.set_ylabel("% of mRNA")

a3 = fig.add_axes([0.84, 0.70, 0.13, 0.26])
v = OX.iloc[::-1]
a3.barh(range(len(v)), v.values, color=[CAT[list(CLASSES).index(k)] for k in v.index], height=0.7)
a3.set_yticks(range(len(v)), v.index, fontsize=5.5)
a3.set(xlabel="OXPHOS subunit mRNA\n(% of cell mRNA)", title="OXPHOS content")
for t, x, y in [("a", 0.01, 0.97), ("b", 0.01, 0.52), ("c", 0.70, 0.97)]:
    fig.text(x, y, t, fontsize=10, fontweight="bold")
fig.savefig(OUT_DIR / "cell_type_deconvolution.pdf")
plt.show()